# 最大池化层

上一章我们发现了一个问题：卷积层虽然用很少的参数（144 个权重）提取了图像特征，但它输出的 16×26×26 特征图，展平后有 10,816 个数值，接入线性层后参数量暴增至接近 70 万，可以算是**参数爆炸**了。

问题的根源在于：卷积层保留了特征图的完整空间尺寸（包括输出通道数）。如果特征图太大，线性层的参数就多得不切实际。

---

**解决方案：池化层**

池化层的思路类似图像压缩。我们把一张 4K 图片缩小到 720p，仍然能看清楚内容。卷积层提取出的特征图也是如此，不需要保留每一个像素的精确位置，只需保留这个区域最重要的信息就够了。

**最大池化层**（MaxPool Layer）的做法，是用一个窗口（**池化核**）分割特征图，每次只保留窗口内数值最大的那个值，其余舍弃。

```{figure} images/pool.png
:align: center
:width: 360px
**图例：最大池化层示意图**
```

一个 2×2 的池化核，将特征图的高度和宽度各缩小一半，面积压缩为原来的 **1/4**。对我们的例子：16×26×26 → 16×13×13，展平后只有 2,704 个数值，参数量从大约 70 万降至大约 17 万，节省了约 75%。

---

**为什么取最大值？**

数值越大说明特征越明显。最大池化保留窗口内最强的响应，目的在于回答：这个区域有没有出现过这个特征？而不是回答：特征精确地在哪个像素？

这带来了额外的好处：**增强了平移不变性**。只要特征出现在池化窗口内的任何位置，都会被捕捉到。比如卷积层检测到一条竖线，无论它在 2×2 窗口的左半边还是右半边，池化后都会得到相同的强响应。

---

**最大池化层的特点总结：**

* **压缩数据**：减小特征图尺寸，大幅降低后续线性层的参数量；
* **增强平移不变性**：特征在池化窗口内小范围移动不影响结果；
* **防止过拟合**：丢弃次要信息，保留最显著特征，降低模型对噪声的敏感度；
* **没有参数**：不增加模型参数，纯粹的计算操作。

In [1]:
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### MNIST 数据集

In [8]:
class MNISTDataset(Dataset):

    def __init__(self, filename, batch_size=1):
        self.filename = filename
        super().__init__(batch_size)

    def load(self):
        with np.load(self.filename, allow_pickle=True) as f:
            self.train_features, self.train_labels = self.preprocess(f['x_train'], f['y_train'])
            self.test_features, self.test_labels = self.preprocess(f['x_test'], f['y_test'])

    @staticmethod
    def preprocess(x, y):
        inputs = x / 255.0
        inputs = np.expand_dims(inputs, axis=1)
        targets = np.zeros((len(y), 10))
        targets[np.arange(len(y)), y] = 1
        return inputs, targets

    def estimate(self, predictions):
        predicted_digits = predictions.data.argmax(axis=1)
        digits = self.labels.argmax(axis=1)
        correct = (predicted_digits == digits).sum()
        return correct / len(self.labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 卷积层

In [11]:
class Convolution(Layer):

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size

        self.weight = Tensor(np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / (in_channels * kernel_size ** 2)))
        self.bias = Tensor(np.zeros(out_channels))

    def forward(self, x: Tensor) -> Tensor:
        batch, in_ch, in_h, in_w = x.data.shape
        out_h = in_h - self.kernel_size + 1
        out_w = in_w - self.kernel_size + 1

        shape = batch, in_ch, out_h, out_w, self.kernel_size, self.kernel_size
        strides = x.data.strides[0], x.data.strides[1], x.data.strides[2], x.data.strides[3], x.data.strides[2], x.data.strides[3]
        patches = np.lib.stride_tricks.as_strided(x.data, shape=shape, strides=strides)
        patches = patches.transpose(0, 2, 3, 1, 4, 5).reshape(batch, out_h * out_w, -1)

        weight = self.weight.data.reshape(self.out_channels, -1)
        output = (patches @ weight.T + self.bias.data)
        output = output.reshape(batch, out_h, out_w, self.out_channels).transpose(0, 3, 1, 2)
        p = Tensor(output)

        def gradient_fn():
            p_grad = p.grad.transpose(0, 2, 3, 1).reshape(batch, out_h * out_w, self.out_channels)
            w_grad = p_grad.reshape(-1, self.out_channels).T @ patches.reshape(-1, patches.shape[-1])
            self.weight.grad += w_grad.reshape(self.weight.data.shape) / batch
            self.bias.grad += p_grad.sum(axis=(0, 1)) / batch

            p_grad_patches = (p_grad @ weight).reshape(batch, out_h, out_w, in_ch, self.kernel_size, self.kernel_size)

            out_h_index = np.arange(out_h)[:, None] + np.arange(self.kernel_size)[None, :]
            out_w_index = np.arange(out_w)[:, None] + np.arange(self.kernel_size)[None, :]

            batch_index = np.arange(batch)[:, None, None, None, None, None]
            ch_index = np.arange(in_ch)[None, None, None, :, None, None]
            h_index = out_h_index[None, :, None, None, :, None]
            w_index = out_w_index[None, None, :, None, None, :]

            grad = np.zeros_like(x.data, dtype=float)
            np.add.at(grad, (batch_index, ch_index, h_index, w_index), p_grad_patches)
            x.grad += grad / batch

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self) -> list[Tensor]:
        return [self.weight, self.bias]

    def __repr__(self) -> str:
        return f"Convolution[weight{self.weight.shape}; bias{self.bias.shape}; kernels={self.out_channels}; kernel_size={self.kernel_size}]"

### 最大池化层

**前向传播**：不同于卷积核，池化核直接分割特征图（不重叠），对每个窗口区域取最大值：

$$
Y_{i,j} = \max_{0 \le m < p,\; 0 \le n < p} X_{i \cdot p + m,\; j \cdot p + n}
$$

其中 $p$ 是池化核边长。输出尺寸为 `height/p` × `width/p`。

代码中同时记录了每个窗口内最大值的位置（`mask` 布尔矩阵），用于反向传播。

**梯度计算**：最大池化没有可学习参数，梯度的作用是把损失信号传回给输入。规则很直观：
- **贡献了最大值的像素**：接收来自输出的完整梯度
- **未贡献最大值的像素**：梯度为 0（它们在前向传播中被丢弃，对输出没有贡献）

实现时，用前向传播保存的 `mask` 做矩阵乘法，把每个输出位置的梯度分配回对应窗口中最大值的位置。

**父节点集合**（parents）：输入值。

**参数列表**（parameters）：无（池化层无可学习参数）。

In [12]:
class MaxPool(Layer):

    def __init__(self, pool_size: int):
        super().__init__()
        self.pool_size = pool_size

    def forward(self, x: Tensor) -> Tensor:
        batch, channel, in_h, in_w = x.data.shape
        # 输出尺寸
        out_h = in_h // self.pool_size
        out_w = in_w // self.pool_size

        # 输出形状：最后扩展两个池化核维度（pool_size, pool_size）
        shape = batch, channel, out_h, out_w, self.pool_size, self.pool_size
        # 各维度的步长：第3、4维乘以 pool_size 实现非重叠滑窗；
        strides = x.data.strides[0], x.data.strides[1], x.data.strides[2] * self.pool_size, x.data.strides[3] * self.pool_size, x.data.strides[2], x.data.strides[3]
        # 根据输出形状和步长创建新的视图（不需要移动数据）
        windows = np.lib.stride_tricks.as_strided(x.data, shape=shape, strides=strides)
        # 在窗口维度 (axis=4,5) 上取最大值，得到池化输出
        output = windows.max(axis=(4, 5))
        # 生成掩码：标记每个窗口中最大值的位置，用于反向传播梯度路由
        mask = windows == output[:, :, :, :, None, None]
        p_tensor = Tensor(output)

        def gradient_fn():
            # 将输出梯度扩展到窗口维度，再乘以掩码，只有最大值位置才接收梯度
            grad = p_tensor.grad[:, :, :, :, None, None] * mask
            x.grad += grad.transpose(0, 1, 2, 4, 3, 5).reshape(batch, channel, in_h, in_w)

        p_tensor.gradient_fn = gradient_fn
        p_tensor.parents = {x}
        return p_tensor

    def __repr__(self) -> str:
        return f"MaxPool[pool_size={self.pool_size}]"

### 展平层

In [13]:
class Flatten(Layer):

    def forward(self, x: Tensor):
        p = Tensor(np.array(x.data.reshape(x.data.shape[0], -1)))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### 丢弃层

In [14]:
class Dropout(Layer):

    def __init__(self, dropout_rate=0.2):
        super().__init__()
        self.dropout_rate = dropout_rate

    def forward(self, x: Tensor):
        if not self.training:
            return x

        mask = np.random.random(x.data.shape) > self.dropout_rate
        p = Tensor(x.data * mask)

        def gradient_fn():
            x.grad += p.grad * mask

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    def __repr__(self):
        return f'Dropout[rate={self.dropout_rate}]'

### ReLU 激活函数层

In [15]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Tanh 激活函数层

In [16]:
class Tanh(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.tanh(x.data))

        def gradient_fn():
            x.grad += a.grad * (1 - a.data ** 2)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Sigmoid 激活函数层

In [17]:
class Sigmoid(Layer):

    def __init__(self, clip_range=(-100, 100)):
        super().__init__()
        self.clip_range = clip_range

    def forward(self, x: Tensor):
        z = np.clip(x.data, self.clip_range[0], self.clip_range[1])
        a = Tensor(1 / (1 + np.exp(-z)))

        def gradient_fn():
            x.grad += a.grad * a.data * (1 - a.data)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Softmax 激活函数层

In [18]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [19]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 交叉熵损失函数

In [20]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

### 二元交叉熵损失函数

In [21]:
class BCELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        clipped = np.clip(p.data, 1e-7, 1 - 1e-7)
        bce = Tensor(-np.mean(y.data * np.log(clipped) + (1 - y.data) * np.log(1 - clipped)))

        def gradient_fn():
            p.grad += (clipped - y.data) / (clipped * (1 - clipped)) / len(y.data)

        bce.gradient_fn = gradient_fn
        bce.parents = {p}
        return bce

### 随机梯度下降优化器

In [22]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [23]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [24]:
LEARNING_RATE = 0.01

### 超参数：批大小

In [25]:
BATCH_SIZE = 2

### 超参数：卷积核大小

In [26]:
KERNEL_SIZE = 3

### 超参数：池化窗口大小

In [27]:
POOL_SIZE = 2

### 超参数：轮次

In [28]:
EPOCHS = 10

### 建模

在卷积层之后、展平层之前插入最大池化层（池化核 2×2）。经过卷积层后特征图为 16×26×26，经过池化层后变为 16×13×13，线性层的输入维度从 10,816 降至 2,704。

In [29]:
dataset = MNISTDataset('tinymnist.npz', BATCH_SIZE)

layer = Sequential([Convolution(1, 16, KERNEL_SIZE),
                    MaxPool(POOL_SIZE),
                    Flatten(),
                    Dropout(),
                    Linear(16 * ((28 - KERNEL_SIZE + 1) // POOL_SIZE) ** 2, 64),
                    ReLU(),
                    Linear(64, 10)])
loss_fn = CELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Convolution[weight=(16, 1, 3, 3); bias=(16,); kernels=16; kernel_size=3]
MaxPool[pool_size=2]
Flatten[]
Dropout[rate=0.2]
Linear[weight(64, 2704); bias(64,)]
ReLU[]
Linear[weight(10, 64); bias(10,)]


### 迭代

In [30]:
model.train(dataset, EPOCHS)

## 验证

### 测试

In [31]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 93.10%


从结果可以观察到：

- **线性层参数量大幅下降**：从大约 70 万降至大约 17 万，训练速度明显提升；
- **准确率基本保持**：丢弃了 75% 的特征图像素，准确率只有轻微下降。

这说明卷积层提取的特征图中，有大量冗余信息。邻近像素的值往往相似，保留最大值就足以代表该区域的特征强度。

``💡 值得注意的是，MNIST 图像本身已经很小（28×28），经过 3×3 卷积后变为 26×26，再 2×2 池化后变为 13×13。这已经接近图像识别有效信息的下限，再继续缩减可能会破坏局部特征，导致准确率明显下降。对于更大的图像，池化层的优势会更加显著。``

## 结束语

这一部分的七个章节，我们一步一步地扩展我们的神经网络训练框架，一直到支持完整的**卷积神经网络（CNN）**：

| 章节 | 组件 | 解决的问题                                 |
|:---:|:---:|:--------------------------------------|
| 第1章 | MNIST 数据集 | 图像数据的加载、归一化、单热编码                      |
| 第2章 | 展平层 | 把多维图像转成线性层能处理的向量                      |
| 第3章 | 丢弃层 | 防止过拟合，提高泛化能力                          |
| 第4章 | Tanh / Sigmoid / Softmax | 更多激活函数，适配不同任务                         |
| 第5章 | 交叉熵损失 | 多分类问题的标准损失函数，解决 Softmax + MSE 的梯度饱和问题 |
| 第6章 | 卷积层 | 捕捉图像局部特征，实现平移不变性，大幅减少参数               |
| 第7章 | 最大池化层 | 压缩特征图尺寸，解决参数爆炸，进一步增强平移不变性             |

图像是网格状数据，**CNN 的核心思路**是：先用卷积层提取局部特征，再用池化层压缩，最后用全连接层分类。这套架构至今仍是图像处理的基础。

下一部分，我们将开始关注**流数据**，比如语言。我们将继续扩展我们的神经网络训练框架，实现对语言和文字的理解，让我们的网络模型开始能“说话”。